# 照片分類整理腳本
讀取 ASSET_LOG → 按 category + keywords + 日期 → 複製到 MAPLAB_Assets 結構。

**執行前必做：**
1. 把 `service_account.json` 上傳到這個 Colab session（左側檔案面板）
2. 確認 `ASSET_LOG_SHEET_ID` 填入正確的 Google Spreadsheet ID
3. 先跑 DRY_RUN=True 確認分類結果，再改 False 正式複製

In [ ]:
# 安裝依賴（Colab 通常已有 google-auth，保險起見重裝）
!pip install -q google-api-python-client google-auth google-auth-httplib2

In [ ]:
# ── 設定（這裡改，不動下方主程式）────────────────────────────────────
ASSET_LOG_SHEET_ID = "YOUR_ASSET_LOG_SHEET_ID"   # ← 填入實際 Spreadsheet ID
ASSET_LOG_SHEET    = "ASSET_LOG"
ROOT_FOLDER_NAME   = "MAPLAB_Assets"
DRY_RUN            = True    # True=預覽前20筆 / False=正式複製
SERVICE_ACCOUNT_FILE = "service_account.json"    # 上傳到 Colab 的 SA 金鑰檔

In [ ]:
import re
from google.oauth2 import service_account
from googleapiclient.discovery import build

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets.readonly",
    "https://www.googleapis.com/auth/drive",
]

FOREIGN_PLACES = [
    "日本", "泰國", "韓國", "越南", "澳門", "香港",
    "馬來西亞", "巴黎", "法國", "義大利", "英國", "美國",
    "德國", "西班牙", "新加坡", "印尼", "菲律賓", "印度",
    "土耳其", "希臘", "荷蘭", "瑞士", "奧地利", "捷克",
    "匈牙利", "波蘭", "葡萄牙", "北歐", "冰島", "加拿大",
    "澳洲", "紐西蘭", "埃及", "摩洛哥", "南非", "阿聯酋",
    "杜拜", "峇里島", "沖繩", "北海道", "東京", "大阪",
    "京都", "首爾", "釜山", "曼谷", "清邁",
]
TAIWAN_PLACES = [
    "台北", "台中", "台南", "高雄", "花蓮", "台東",
    "宜蘭", "苗栗", "彰化", "南投", "嘉義", "屏東",
    "澎湖", "金門", "馬祖", "基隆", "新竹", "桃園",
    "新北", "阿里山", "墾丁", "日月潭", "太魯閣",
]


def extract_date(original_name):
    patterns = [
        r"20(\d{2})[-_](\d{2})[-_](\d{2})",
        r"20(\d{2})(\d{2})(\d{2})",
    ]
    for pat in patterns:
        m = re.search(pat, original_name)
        if m:
            yy, mm, dd = m.group(1), m.group(2), m.group(3)
            if 1 <= int(mm) <= 12 and 1 <= int(dd) <= 31:
                return f"20{yy}", mm, dd
    return None, None, None


def date_label(original_name, ym_only=False):
    y, m, d = extract_date(original_name)
    if y is None:
        return "日期不明"
    return f"{y}-{m}" if ym_only else f"{y}-{m}-{d}"


def classify(row):
    category = (row.get("category") or "").strip()
    keywords = (row.get("keywords") or "").strip()
    original = (row.get("original_name") or "").strip()
    kw = keywords

    if "外燴" in category:
        date_str = date_label(original, ym_only=False)
        if "生日" in kw:
            return ["外燴", "生日派對", date_str]
        if any(w in kw for w in ["婚", "婚禮", "婚宴"]):
            return ["外燴", "婚禮", date_str]
        if any(w in kw for w in ["企業", "開幕", "尾牙", "春酒"]):
            return ["外燴", "企業活動", date_str]
        return ["外燴", "其他", date_str]

    if "旅遊" in category:
        date_ym = date_label(original, ym_only=True)
        for place in FOREIGN_PLACES:
            if place in kw:
                return ["旅遊", "國外", f"{date_ym} {place}"]
        for place in TAIWAN_PLACES:
            if place in kw:
                return ["旅遊", "台灣", f"{date_ym} {place}"]
        if any(w in kw for w in ["海", "玩水", "沙灘", "潮間帶"]):
            return ["旅遊", "海邊玩水", date_ym]
        if "露營" in kw:
            return ["旅遊", "露營", date_ym]
        return ["旅遊", "其他", date_ym]

    if "日常" in category:
        date_ym = date_label(original, ym_only=True)
        return ["日常", date_ym]

    return ["error"]


_folder_cache = {}


def get_or_create_folder(service, name, parent_id):
    cache_key = f"{parent_id}/{name}"
    if cache_key in _folder_cache:
        return _folder_cache[cache_key]
    query = (
        f"name='{name}' and mimeType='application/vnd.google-apps.folder'"
        f" and '{parent_id}' in parents and trashed=false"
    )
    resp = service.files().list(q=query, fields="files(id)").execute()
    files = resp.get("files", [])
    if files:
        fid = files[0]["id"]
    else:
        meta = {
            "name": name,
            "mimeType": "application/vnd.google-apps.folder",
            "parents": [parent_id],
        }
        fid = service.files().create(body=meta, fields="id").execute()["id"]
    _folder_cache[cache_key] = fid
    return fid


def ensure_folder_path(service, root_id, path_parts):
    current = root_id
    for part in path_parts:
        current = get_or_create_folder(service, part, current)
    return current


def copy_file(service, file_id, dest_folder_id, new_name):
    body = {"name": new_name, "parents": [dest_folder_id]}
    return service.files().copy(fileId=file_id, body=body, fields="id").execute()["id"]


# ── 主流程 ────────────────────────────────────────────────────────────
creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)
sheets_svc = build("sheets", "v4", credentials=creds)
drive_svc  = build("drive",  "v3", credentials=creds)

result = (
    sheets_svc.spreadsheets()
    .values()
    .get(spreadsheetId=ASSET_LOG_SHEET_ID, range=f"{ASSET_LOG_SHEET}!A1:Z")
    .execute()
)
rows = result.get("values", [])
if not rows:
    print("ASSET_LOG 是空的，結束。")
else:
    headers = [h.strip() for h in rows[0]]
    records = [dict(zip(headers, r)) for r in rows[1:]]
    print(f"共讀到 {len(records)} 筆")

    if DRY_RUN:
        print("\n[DRY_RUN] 前 20 筆分類預覽：")
        print(f"{'original_name':<30} {'seo_name':<40} 目標路徑")
        print("-" * 100)
        for rec in records[:20]:
            path = classify(rec)
            print(f"{rec.get('original_name',''):<30} {rec.get('seo_name',''):<40} {'/'.join(path)}")
        print("\nDRY_RUN 完成。設 DRY_RUN=False 後重執行才實際複製。")
    else:
        root_id = get_or_create_folder(drive_svc, ROOT_FOLDER_NAME, "root")
        print(f"根資料夾 {ROOT_FOLDER_NAME} id={root_id}")
        ok, fail = 0, 0
        for i, rec in enumerate(records):
            file_id  = rec.get("file_id", "").strip()
            seo_name = rec.get("seo_name", "").strip()
            if not file_id or not seo_name:
                print(f"[SKIP] 第 {i+2} 行缺 file_id 或 seo_name")
                continue
            path_parts = classify(rec)
            try:
                dest = ensure_folder_path(drive_svc, root_id, path_parts)
                copy_file(drive_svc, file_id, dest, seo_name)
                ok += 1
            except Exception as e:
                print(f"[FAIL] {rec.get('original_name','')} → {'/'.join(path_parts)}: {e}")
                fail += 1
            if (i + 1) % 100 == 0:
                print(f"進度：{i+1}/{len(records)}（成功 {ok} / 失敗 {fail}）")
        print(f"\n完成！成功 {ok}，失敗 {fail}，共 {len(records)} 筆")